In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
import torchvision.transforms as transforms
from torchvision import datasets
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# DEVICE
# =========================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# =========================================================
# DATA AUGMENTATION & TRANSFORMS
# =========================================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =========================================================
# LOAD DATASET
# =========================================================

full_dataset = datasets.ImageFolder(
    root="dataset",
    transform=eval_transform
)

dataset_size = len(full_dataset)
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size

generator = torch.Generator().manual_seed(42)
indices = torch.randperm(dataset_size, generator=generator).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

train_dataset = Subset(
    datasets.ImageFolder(root="dataset", transform=train_transform),
    train_indices
)

val_dataset = Subset(
    datasets.ImageFolder(root="dataset", transform=eval_transform),
    val_indices
)

test_dataset = Subset(
    datasets.ImageFolder(root="dataset", transform=eval_transform),
    test_indices
)

# =========================================================
# DATALOADERS
# =========================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

# =========================================================
# INFO
# =========================================================

num_classes = len(full_dataset.classes)

print("Classes:", full_dataset.classes)
print("Number of classes:", num_classes)
print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))


Using device: cpu
Classes: ['Angry', 'Other', 'Sad', 'happy']
Number of classes: 4
Train size: 700
Validation size: 150
Test size: 150


In [ ]:
class BasicConv(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=False, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        return torch.relu(self.bn(self.conv(x)))


class InceptionA(nn.Module):
    def __init__(self, in_channels, pool_features):
        super().__init__()
        self.branch1 = BasicConv(in_channels, 64, kernel_size=1)
        self.branch2 = nn.Sequential(
            BasicConv(in_channels, 48, kernel_size=1),
            BasicConv(48, 64, kernel_size=3, padding=1)
        )
        self.branch3 = nn.Sequential(
            BasicConv(in_channels, 64, kernel_size=1),
            BasicConv(64, 96, kernel_size=3, padding=1),
            BasicConv(96, 96, kernel_size=3, padding=1)
        )
        self.branch4 = nn.Sequential(
            nn.AvgPool2d(kernel_size=3, stride=1, padding=1),
            BasicConv(in_channels, pool_features, kernel_size=1)
        )

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], 1)


class InceptionB(nn.Module):
    def __init__(self, in_channels, channels_7x7):
        super().__init__()
        c7 = channels_7x7
        self.branch1 = BasicConv(in_channels, 192, kernel_size=1)
        self.branch2 = nn.Sequential(
            BasicConv(in_channels, c7, kernel_size=1),
            BasicConv(c7, c7, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv(c7, 192, kernel_size=(7, 1), padding=(3, 0))
        )
        self.branch3 = nn.Sequential(
            BasicConv(in_channels, c7, kernel_size=1),
            BasicConv(c7, c7, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv(c7, c7, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv(c7, c7, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv(c7, 192, kernel_size=(1, 7), padding=(0, 3))
        )
        self.branch4 = nn.Sequential(
            nn.AvgPool2d(kernel_size=3, stride=1, padding=1),
            BasicConv(in_channels, 192, kernel_size=1)
        )

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], 1)


class InceptionC(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.branch1 = BasicConv(in_channels, 320, kernel_size=1)
        self.branch2_reduce = BasicConv(in_channels, 384, kernel_size=1)
        self.branch2a = BasicConv(384, 384, kernel_size=(1, 3), padding=(0, 1))
        self.branch2b = BasicConv(384, 384, kernel_size=(3, 1), padding=(1, 0))
        self.branch3_reduce = nn.Sequential(
            BasicConv(in_channels, 448, kernel_size=1),
            BasicConv(448, 384, kernel_size=3, padding=1)
        )
        self.branch3a = BasicConv(384, 384, kernel_size=(1, 3), padding=(0, 1))
        self.branch3b = BasicConv(384, 384, kernel_size=(3, 1), padding=(1, 0))
        self.branch4 = nn.Sequential(
            nn.AvgPool2d(kernel_size=3, stride=1, padding=1),
            BasicConv(in_channels, 192, kernel_size=1)
        )

    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2_reduce(x)
        b2 = torch.cat([self.branch2a(b2), self.branch2b(b2)], 1)
        b3 = self.branch3_reduce(x)
        b3 = torch.cat([self.branch3a(b3), self.branch3b(b3)], 1)
        b4 = self.branch4(x)
        return torch.cat([b1, b2, b3, b4], 1)


class ReductionA(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.branch1 = BasicConv(in_channels, 384, kernel_size=3, stride=2, padding=1)
        self.branch2 = nn.Sequential(
            BasicConv(in_channels, 64, kernel_size=1),
            BasicConv(64, 96, kernel_size=3, padding=1),
            BasicConv(96, 96, kernel_size=3, stride=2, padding=1)
        )
        self.branch3 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], 1)


class ReductionB(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.branch1 = nn.Sequential(
            BasicConv(in_channels, 192, kernel_size=1),
            BasicConv(192, 320, kernel_size=3, stride=2, padding=1)
        )
        self.branch2 = nn.Sequential(
            BasicConv(in_channels, 192, kernel_size=1),
            BasicConv(192, 192, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv(192, 192, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv(192, 192, kernel_size=3, stride=2, padding=1)
        )
        self.branch3 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], 1)


class AuxClassifier(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.conv1 = BasicConv(in_channels, 128, kernel_size=1)
        self.conv2 = BasicConv(128, 768, kernel_size=4)
        self.fc = nn.Linear(768, num_classes)

    def forward(self, x):
        x = self.pool(x)
        x = self.conv1(x)
        x = self.conv2(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


class InceptionV3(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.stem = nn.Sequential(
            BasicConv(3, 32, kernel_size=3, stride=2, padding=1),
            BasicConv(32, 32, kernel_size=3, padding=1),
            BasicConv(32, 64, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            BasicConv(64, 80, kernel_size=1),
            BasicConv(80, 192, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        self.inceptionA1 = InceptionA(192, pool_features=32)
        self.inceptionA2 = InceptionA(256, pool_features=64)
        self.inceptionA3 = InceptionA(288, pool_features=64)
        self.reductionA  = ReductionA(288)
        self.inceptionB1 = InceptionB(768, channels_7x7=128)
        self.inceptionB2 = InceptionB(768, channels_7x7=160)
        self.inceptionB3 = InceptionB(768, channels_7x7=160)
        self.inceptionB4 = InceptionB(768, channels_7x7=192)
        self.aux         = AuxClassifier(768, num_classes)
        self.reductionB  = ReductionB(768)
        self.inceptionC1 = InceptionC(1280)
        self.inceptionC2 = InceptionC(2048)
        self.pool        = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout     = nn.Dropout(0.5)
        self.fc          = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.inceptionA1(x)
        x = self.inceptionA2(x)
        x = self.inceptionA3(x)
        x = self.reductionA(x)
        x = self.inceptionB1(x)
        x = self.inceptionB2(x)
        x = self.inceptionB3(x)
        x = self.inceptionB4(x)
        aux_out = self.aux(x)
        x = self.reductionB(x)
        x = self.inceptionC1(x)
        x = self.inceptionC2(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x), aux_out


inception_model = InceptionV3(num_classes=num_classes).to(device)

inception_optimizer = optim.Adam(inception_model.parameters(), lr=0.001)


def train_inception(model, optimizer, train_loader, val_loader, epochs):
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        total = 0
        loop = tqdm(train_loader)

        for images, labels in loop:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            main_out, aux_out = model(images)
            loss = criterion(main_out, labels) + 0.4 * criterion(aux_out, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(main_out, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
            loop.set_postfix(loss=loss.item())

        train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct / total
        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                main_out, _ = model(images)
                loss = criterion(main_out, labels)
                val_loss += loss.item()
                _, predicted = torch.max(main_out, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        val_accuracy = 100 * val_correct / val_total
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.2f}%")
        print(f"Val Loss: {val_loss:.4f} | Val Accuracy: {val_accuracy:.2f}%")

    return train_losses, val_losses, train_accuracies, val_accuracies


def evaluate_inception(model, test_loader, class_names):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            main_out, _ = model(images)
            _, predicted = torch.max(main_out, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    cm = compute_confusion_matrix(all_labels, all_preds, len(class_names))
    precision, recall, f1, weighted_precision, weighted_recall, weighted_f1 = compute_precision_recall_f1(cm)
    accuracy = 100 * np.mean(np.array(all_preds) == np.array(all_labels))

    print("\n================ RESULTS ================\n")
    print(f"Accuracy  : {accuracy:.2f}%")
    print(f"Weighted Precision : {weighted_precision:.4f}")
    print(f"Weighted Recall    : {weighted_recall:.4f}")
    print(f"Weighted F1 Score  : {weighted_f1:.4f}\n")
    print("Classification Report:\n")
    print(format_classification_report(cm, class_names))

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    fig.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel='Actual',
        xlabel='Predicted',
        title='Confusion Matrix'
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
    thresh = cm.max() / 2.0 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = 'white' if cm[i, j] > thresh else 'black'
            ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center', color=color)
    plt.tight_layout()
    plt.show()


inception_train_losses, inception_val_losses, inception_train_acc, inception_val_acc = train_inception(
    inception_model,
    inception_optimizer,
    train_loader,
    val_loader,
    epochs=20
)

evaluate_inception(inception_model, test_loader, full_dataset.classes)

torch.save(inception_model.state_dict(), "inceptionv3_scratch.pth")

print("\nInceptionV3 Model Saved!")
